# 04 — Fair Logistic Regression

Trains the custom `FairLogisticRegression` class over a sweep of
fairness-penalty values `λ_f ∈ {0, 0.1, 0.5, 1.0, 2.0, 5.0}` and shows
the resulting accuracy / segment-disparity Pareto frontier.

**Maps to paper Sections 8 and 9.3. Produces Figure 2.**


## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


## 2. Load prepared data and segments

In [ ]:
DATA = Path("../outputs/tables")
X = pd.read_csv(DATA / "X_features.csv")
y = pd.read_csv(DATA / "y_target.csv").squeeze("columns")
segments_df = pd.read_csv(DATA / "segments.csv")
segments = segments_df["Contract"].values

X_np = X.values.astype(np.float64)
y_np = y.values.astype(np.float64)
print(f"X: {X_np.shape}, y: {y_np.shape}, segments: {len(segments)}")


## 3. Sweep λ_fairness

In [ ]:
from sklearn.metrics import accuracy_score
from src.fair_regression import FairLogisticRegression
from src.metrics import mean_error_by_segment

lambda_f_values = [0, 0.1, 0.5, 1.0, 2.0, 5.0]
results_fair = []

for lf in lambda_f_values:
    print(f"  Training with λ_f = {lf}...")
    model = FairLogisticRegression(
        lambda_complexity=0.1,
        lambda_fairness=lf,
        segments=segments,
    )
    model.fit(X_np, y_np)
    y_pred_proba = model.predict_proba(X_np)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    acc = accuracy_score(y_np, y_pred)
    errors = mean_error_by_segment(y_np, y_pred_proba, segments)
    disparity = max(errors.values()) - min(errors.values())

    results_fair.append({
        "lambda_f": lf,
        "accuracy": acc,
        "disparity": disparity,
    })

fair_df = pd.DataFrame(results_fair)
fair_df.to_csv("../outputs/tables/fairness_lambda_sweep.csv", index=False)
print()
print(fair_df.round(4).to_string(index=False))


## 4. Figure 2 — Fairness–Accuracy Trade-off

In [ ]:
from src.visualizations import plot_fairness_accuracy_tradeoff, savefig

fig = plot_fairness_accuracy_tradeoff(fair_df)
savefig(fig, "../outputs/figures/fig2_fairness_accuracy_tradeoff.png")
fig.show()


## 5. Train the final fair model at λ_f = 2.0

This is the model used for the segment-level analysis in notebook 05.


In [ ]:
fair_model = FairLogisticRegression(
    lambda_complexity=0.1,
    lambda_fairness=2.0,
    segments=segments,
)
fair_model.fit(X_np, y_np)

fair_pred_proba = fair_model.predict_proba(X_np)[:, 1]
fair_pred_class = (fair_pred_proba >= 0.5).astype(int)

# Build the analysis dataframe used by notebook 05
df_analysis = segments_df.copy()
df_analysis["y_true"] = y_np
df_analysis["fair_pred_class"] = fair_pred_class
df_analysis["fair_pred_proba"] = fair_pred_proba
df_analysis["fair_residual"]   = y_np - fair_pred_proba

# Also produce baseline (Gradient Boosting) predictions for comparison.
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_np, y_np)
df_analysis["y_pred_proba"]   = gb.predict_proba(X_np)[:, 1]
df_analysis["y_pred_class"]   = (df_analysis["y_pred_proba"] >= 0.5).astype(int)
df_analysis["residual"]       = y_np - df_analysis["y_pred_proba"]

df_analysis.to_csv("../outputs/tables/df_analysis.csv", index=False)
print(f"df_analysis: {df_analysis.shape}")
df_analysis.head()


## Key takeaway

As `λ_f` rises from 0 to 2.0, the maximum segment disparity drops from
~0.161 to ~0.136 while accuracy actually *improves* slightly (from
~0.809 to ~0.813). This is the Pareto improvement claim in Section 9.3:
fairness regularization can act as a useful inductive bias and reduce
disparity without harming overall accuracy.

Continue with [05_segment_analysis.ipynb](./05_segment_analysis.ipynb).
